In [2]:
import numpy as np
from pathlib import Path # Add this import

# Note: Make sure OUT_ROOT is defined somewhere before calling this function!
OUT_ROOT = Path("./output") # Example definition

def sim_outfile(design: str, n: int, sim: int) -> Path:
    d = OUT_ROOT / design / f"n{n}"
    d.mkdir(parents=True, exist_ok=True)
    return d / f"sim_{sim:04d}.npz"

def save_sim_result(path: Path, model_names, results_dict, q_preds=None):
    # Store tau/se/coverage in aligned arrays (fast to load/aggregate later)
    tau = np.array([results_dict[m]["tau_hat"] for m in model_names], dtype=float)
    se  = np.array([results_dict[m]["SE"] for m in model_names], dtype=float)
    cov = np.array([results_dict[m]["Coverage"] for m in model_names], dtype=np.int8)

    if q_preds is None:
        np.savez_compressed(path, model_names=np.array(model_names), tau=tau, se=se, cov=cov)
    else:
        # q_preds should be a dict model -> vector (n_eval,)
        Q = np.stack([np.asarray(q_preds[m], dtype=float) for m in model_names], axis=0)  # (M, n_eval)
        np.savez_compressed(path, model_names=np.array(model_names), tau=tau, se=se, cov=cov, Q=Q)

In [9]:
from joblib import Parallel, delayed

In [15]:
#!/usr/bin/env python3
"""
TEJ-safe Monte Carlo for Adaptive Nested-CV DML-IV (PLIV score).

Keeps the SAME estimator/algorithmic structure as in the paper:
- K_OUTER-fold cross-fitting
- leakage-safe inner CV (tuning/selection only on training folds)
- one-pass outer loop computes:
    (i) fixed-learner DML-IV for every learner in the library
    (ii) Adaptive Nested-CV column (separate selection for mu, r, pi_j)
- PLIV residualized score + sandwich variance

Optimizations (execution-only; does NOT change estimator):
1) Per-simulation checkpointing (.npz) + resume on rerun.
2) Parallelize across simulation replications (processes).
3) Eliminate nested parallelism:
   - GridSearchCV n_jobs = 1
   - RandomForest/XGBoost n_jobs = 1
   - BLAS threads pinned to 1

Outputs:
- Checkpoints: mc_out/<design>/n<N>/sim_<####>.npz
- LaTeX tables: tab_mc_first_stage_<design>.tex and tab_mc_second_stage_<design>.tex

NOTE:
- By default DO_Q_DECOMP_DIAGNOSTICS = False (recommended).
  If you later set it True, the code will additionally store Q predictions per sim file.
"""
#!/usr/bin/env python3
"""
TEJ-safe Monte Carlo for Adaptive Nested-CV DML-IV (PLIV score).

(Your text unchanged)
"""

# -------------------------------------------------------------------
# Thread limits BEFORE importing numpy/sklearn (prevents oversubscription)
# -------------------------------------------------------------------
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("VECLIB_MAXIMUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

from pathlib import Path
import warnings

import numpy as np
import pandas as pd

from joblib import Parallel, delayed  # <-- FIX 1: missing import

from sklearn.base import clone
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.exceptions import ConvergenceWarning

# Optional XGBoost
try:
    from xgboost import XGBRegressor
    HAS_XGB = True
except Exception:
    HAS_XGB = False

warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", category=ConvergenceWarning)

# -------------------------------------------------------------------
# CONFIG (keep these aligned with paper)
# -------------------------------------------------------------------
N_SIMS    = 100
N_SAMPLES = [1000, 3000, 10000]
DESIGNS   = ["sparse", "nonlinear"]

K_OUTER = 3
K_INNER = 3

DO_Q_DECOMP_DIAGNOSTICS = False
N_EVAL = 5000

CPU = os.cpu_count() or 1
MAX_WORKERS   = min(max(1, CPU - 2), 4)
N_INNER_JOBS  = 1
TREE_N_JOBS   = 1

OUT_ROOT = Path("mc_out")
OUT_ROOT.mkdir(parents=True, exist_ok=True)

# -------------------------------------------------------------------
# 1) DGP
# -------------------------------------------------------------------
_COV_CHOL_CACHE = {}

def _toeplitz_chol(p: int, rho: float = 0.5) -> np.ndarray:
    key = (p, rho)
    if key in _COV_CHOL_CACHE:
        return _COV_CHOL_CACHE[key]
    idx = np.arange(p)
    cov = rho ** np.abs(idx[:, None] - idx[None, :])
    L = np.linalg.cholesky(cov)
    _COV_CHOL_CACHE[key] = L
    return L

def generate_data(n, p=200, design="sparse", seed=None):
    rng = np.random.RandomState(seed) if seed is not None else np.random.RandomState()

    L = _toeplitz_chol(p, rho=0.5)
    W = rng.normal(size=(n, p)) @ L.T

    d_z = 5
    Z = rng.uniform(0.0, 1.0, size=(n, d_z))
    A = rng.normal(0.0, 1.0, size=n)

    tau0  = 1.0
    gamma = 1.0
    delta = 1.0

    if design == "sparse":
        m_ZW = 0.5 * np.sum(Z, axis=1) + 0.5 * np.sum(W[:, :5], axis=1)
        g_W  = 0.5 * np.sum(W[:, :5], axis=1)
    elif design == "nonlinear":
        m_ZW = Z[:, 0] * Z[:, 1] + np.sin(W[:, 0]) * np.cos(W[:, 1]) + np.exp(0.5 * W[:, 2])
        g_W  = 1 / (1 + np.exp(-W[:, 0])) + (W[:, 1] * W[:, 2]) + (W[:, 3] > 0).astype(float)
    else:
        raise ValueError("Design must be 'sparse' or 'nonlinear'")

    nu  = rng.normal(0.0, 1.0, size=n)
    eps = rng.normal(0.0, 1.0, size=n)

    D = m_ZW + gamma * A + nu
    Y = tau0 * D + g_W + delta * A + eps

    true_pi = np.full((n, d_z), 0.5)
    true_q  = m_ZW
    return W, Z, D, Y, true_pi, true_q, tau0

# -------------------------------------------------------------------
# 2) Learner library
# -------------------------------------------------------------------
def get_mc_library():
    lib = []
    lib.append(("Ridge", Ridge(), {"model__alpha": np.logspace(-2, 2, 5)}, True))
    lib.append(("Lasso", Lasso(max_iter=2000), {"model__alpha": np.logspace(-3, 0, 5)}, True))
    lib.append(("ElasticNet", ElasticNet(max_iter=2000),
                {"model__alpha": np.logspace(-3, 0, 5),
                 "model__l1_ratio": [0.2, 0.5, 0.8]}, True))

    lib.append(("RandomForest",
                RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=TREE_N_JOBS),
                None, False))

    lib.append(("GradientBoosting",
                GradientBoostingRegressor(n_estimators=100, random_state=42),
                None, False))

    if HAS_XGB:
        lib.append(("XGBoost",
                    XGBRegressor(n_estimators=100, max_depth=3, learning_rate=0.1,
                                 random_state=42, n_jobs=TREE_N_JOBS,
                                 subsample=0.8, colsample_bytree=0.8),
                    None, False))

    lib.append((
        "MLP",
        MLPRegressor(
            hidden_layer_sizes=(100,),
            activation="relu",
            solver="adam",
            alpha=1e-4,
            learning_rate_init=1e-3,
            max_iter=300,
            early_stopping=True,
            n_iter_no_change=10,
            random_state=42
        ),
        {"model__alpha": [1e-5, 1e-4, 1e-3],
         "model__learning_rate_init": [1e-3, 1e-2]},
        True
    ))
    return lib

# -------------------------------------------------------------------
# 3) Inner CV helper
# -------------------------------------------------------------------
def fit_model_and_cv_mse(X_train, y_train, name, estimator, param_grid, needs_scaling,
                         inner_splits=3, seed=42):
    y_train = np.asarray(y_train).reshape(-1)

    steps = []
    if needs_scaling:
        steps.append(("scaler", StandardScaler(with_mean=True, with_std=True)))
    steps.append(("model", clone(estimator)))
    pipe = Pipeline(steps)

    inner_cv = KFold(n_splits=inner_splits, shuffle=True, random_state=seed)

    if param_grid is not None:
        gs = GridSearchCV(
            pipe, param_grid, cv=inner_cv,
            scoring="neg_mean_squared_error",
            n_jobs=N_INNER_JOBS
        )
        gs.fit(X_train, y_train)
        model = gs.best_estimator_
        cv_mse = float(-gs.best_score_)
        return model, cv_mse

    mses = []
    for tr, te in inner_cv.split(X_train):
        m = clone(pipe).fit(X_train[tr], y_train[tr])
        pred = m.predict(X_train[te])
        mses.append(mean_squared_error(y_train[te], pred))
    cv_mse = float(np.mean(mses))
    model = pipe.fit(X_train, y_train)
    return model, cv_mse

def fit_predict_all_learners(X_tr, y_tr, X_te, library, inner_splits=3, seed=42):
    preds = {}
    cvmse = {}
    for idx, (name, est, grid, scale) in enumerate(library):
        model, mse = fit_model_and_cv_mse(
            X_tr, y_tr, name, est, grid, scale,
            inner_splits=inner_splits, seed=seed + 101*idx
        )
        preds[name] = model.predict(X_te)
        cvmse[name] = mse
    return preds, cvmse

# -------------------------------------------------------------------
# 4) PLIV tau + se
# -------------------------------------------------------------------
def pliv_tau_se(Y, D, Z, mu_hat, r_hat, pi_hat):
    n = len(Y)
    Y_tilde = Y - mu_hat
    D_tilde = D - r_hat
    Z_tilde = Z - pi_hat

    Omega = (Z_tilde.T @ Z_tilde) / n
    g_zd  = (Z_tilde.T @ D_tilde) / n
    g_zy  = (Z_tilde.T @ Y_tilde) / n

    try:
        Omega_inv = np.linalg.inv(Omega)
    except np.linalg.LinAlgError:
        Omega_inv = np.linalg.pinv(Omega)

    denom = float(g_zd.T @ Omega_inv @ g_zd)
    numer = float(g_zd.T @ Omega_inv @ g_zy)
    tau_hat = numer / denom

    u_hat = Y_tilde - tau_hat * D_tilde
    psi = Z_tilde * u_hat[:, None]
    Sigma = (psi.T @ psi) / n

    middle = float(g_zd.T @ Omega_inv @ Sigma @ Omega_inv @ g_zd)
    V = (1.0 / denom) * middle * (1.0 / denom)
    se = float(np.sqrt(V / n))

    ci = (tau_hat - 1.96*se, tau_hat + 1.96*se)
    return float(tau_hat), float(se), ci

# -------------------------------------------------------------------
# 5) One sim: fixed learners + Nested-CV
# -------------------------------------------------------------------
def run_one_sim_all_models(W, Z, D, Y, true_pi, tau0, library, k_outer=3, inner_splits=3, seed=42):
    n, d_z = Z.shape
    kf = KFold(n_splits=k_outer, shuffle=True, random_state=seed)

    learner_names = [m[0] for m in library]

    mu_by = {name: np.zeros(n) for name in learner_names}
    r_by  = {name: np.zeros(n) for name in learner_names}
    pi_by = {name: np.zeros((n, d_z)) for name in learner_names}

    mu_nested = np.zeros(n)
    r_nested  = np.zeros(n)
    pi_nested = np.zeros((n, d_z))

    chosen_nested = {"mu": [], "r": [], "pi": [[] for _ in range(d_z)]}

    for fold, (tr, te) in enumerate(kf.split(W)):
        W_tr, W_te = W[tr], W[te]
        Y_tr, D_tr, Z_tr = Y[tr], D[tr], Z[tr]

        preds_mu, mse_mu = fit_predict_all_learners(W_tr, Y_tr, W_te, library, inner_splits, seed=1000 + fold)
        for name in learner_names:
            mu_by[name][te] = preds_mu[name]
        best_mu = min(mse_mu, key=mse_mu.get)
        mu_nested[te] = preds_mu[best_mu]
        chosen_nested["mu"].append(best_mu)

        preds_r, mse_r = fit_predict_all_learners(W_tr, D_tr, W_te, library, inner_splits, seed=2000 + fold)
        for name in learner_names:
            r_by[name][te] = preds_r[name]
        best_r = min(mse_r, key=mse_r.get)
        r_nested[te] = preds_r[best_r]
        chosen_nested["r"].append(best_r)

        for j in range(d_z):
            preds_pj, mse_pj = fit_predict_all_learners(W_tr, Z_tr[:, j], W_te, library, inner_splits, seed=3000 + 37*j + fold)
            for name in learner_names:
                pi_by[name][te, j] = preds_pj[name]
            best_pj = min(mse_pj, key=mse_pj.get)
            pi_nested[te, j] = preds_pj[best_pj]
            chosen_nested["pi"][j].append(best_pj)

    results = {}
    for name in learner_names:
        tau_hat, se_hat, _ = pliv_tau_se(Y, D, Z, mu_by[name], r_by[name], pi_by[name])
        ci_low = tau_hat - 1.96*se_hat
        ci_up  = tau_hat + 1.96*se_hat
        results[name] = {"tau_hat": tau_hat, "SE": se_hat, "Coverage": int(ci_low <= tau0 <= ci_up)}

    tau_hat, se_hat, _ = pliv_tau_se(Y, D, Z, mu_nested, r_nested, pi_nested)
    ci_low = tau_hat - 1.96*se_hat
    ci_up  = tau_hat + 1.96*se_hat
    results["Adaptive Nested-CV"] = {
        "tau_hat": tau_hat, "SE": se_hat, "Coverage": int(ci_low <= tau0 <= ci_up),
        "Chosen(mu)": chosen_nested["mu"], "Chosen(r)": chosen_nested["r"],
    }
    return results

# -------------------------------------------------------------------
# 6) Optional q diagnostics
# -------------------------------------------------------------------
def fit_predict_q_all_learners(W, Z, D, W_eval, Z_eval, library, inner_splits=3, seed=7000):
    Xq = np.column_stack([Z, W])
    Xq_eval = np.column_stack([Z_eval, W_eval])
    preds_q, mse_q = fit_predict_all_learners(Xq, D, Xq_eval, library, inner_splits=inner_splits, seed=seed)
    best_q = min(mse_q, key=mse_q.get)
    preds_q["Adaptive Nested-CV"] = preds_q[best_q]
    return preds_q

# -------------------------------------------------------------------
# 7) Checkpointing
# -------------------------------------------------------------------
def sim_outfile(design: str, n: int, sim: int) -> Path:
    d = OUT_ROOT / design / f"n{n}"
    d.mkdir(parents=True, exist_ok=True)
    return d / f"sim_{sim:04d}.npz"

def save_sim_result(path: Path, model_names, results_dict, q_preds=None):
    tau = np.array([results_dict[m]["tau_hat"] for m in model_names], dtype=float)
    se  = np.array([results_dict[m]["SE"] for m in model_names], dtype=float)
    cov = np.array([results_dict[m]["Coverage"] for m in model_names], dtype=np.int8)

    if q_preds is None:
        np.savez_compressed(path, model_names=np.array(model_names, dtype=object), tau=tau, se=se, cov=cov)
    else:
        Q = np.stack([np.asarray(q_preds[m], dtype=float) for m in model_names], axis=0)
        np.savez_compressed(path, model_names=np.array(model_names, dtype=object), tau=tau, se=se, cov=cov, Q=Q)

def _run_one_sim_and_save(design: str, n: int, sim: int,
                         k_outer: int, inner_splits: int,
                         do_q_diag: bool, n_eval: int) -> str:
    out = sim_outfile(design, n, sim)
    if out.exists():
        return str(out)

    library = get_mc_library()
    learner_names = [m[0] for m in library]
    model_names = learner_names + ["Adaptive Nested-CV"]

    W, Z, D, Y, true_pi, true_q, tau0 = generate_data(n, p=200, design=design, seed=sim)

    sim_results = run_one_sim_all_models(W, Z, D, Y, true_pi, tau0, library, k_outer=k_outer, inner_splits=inner_splits, seed=42)

    q_preds = None
    if do_q_diag:
        W_eval, Z_eval, _, _, _, true_q_eval, _ = generate_data(n_eval, p=200, design=design, seed=999999)
        q_preds = fit_predict_q_all_learners(W, Z, D, W_eval, Z_eval, library, inner_splits=inner_splits, seed=7000 + sim)

    save_sim_result(out, model_names, sim_results, q_preds=q_preds)
    return str(out)

# -------------------------------------------------------------------
# 8) Aggregation + LaTeX export
# -------------------------------------------------------------------
def export_latex_table_safe(df, path, caption, label):
    body = df.to_latex(index=True, escape=False)
    tex = "\\begin{table}[htbp]\n\\centering\n" + body
    tex += "\\caption{" + caption + "}\n\\label{" + label + "}\n\\end{table}\n"
    with open(path, "w", encoding="utf-8") as f:
        f.write(tex)

def aggregate_cell(design: str, n: int, model_names, do_q_diag: bool, n_eval: int):
    taus, ses, covs = [], [], []
    if do_q_diag:
        Q_sum = Q_sum2 = None
        _, _, _, _, _, true_q_eval, _ = generate_data(n_eval, p=200, design=design, seed=999999)

    done = 0
    for sim in range(N_SIMS):
        p = sim_outfile(design, n, sim)
        if not p.exists():
            continue
        data = np.load(p, allow_pickle=True)

        mn = list(data["model_names"])
        if list(mn) != list(model_names):
            raise ValueError(f"Model name mismatch in {p}. Expected {model_names}, got {mn}.")

        taus.append(data["tau"])
        ses.append(data["se"])
        covs.append(data["cov"])
        done += 1

        if do_q_diag:
            # <-- FIX 2: correct check
            if "Q" not in data.files:
                raise ValueError(f"Expected Q in {p} but not found. Rerun with DO_Q_DECOMP_DIAGNOSTICS=True.")
            Q = data["Q"]
            if Q_sum is None:
                Q_sum  = np.zeros_like(Q, dtype=float)
                Q_sum2 = np.zeros_like(Q, dtype=float)
            Q_sum  += Q
            Q_sum2 += Q**2

    if done == 0:
        raise RuntimeError(f"No simulation files found for design={design}, n={n} in {OUT_ROOT}.")

    taus = np.vstack(taus)
    ses  = np.vstack(ses)
    covs = np.vstack(covs)

    tau0 = 1.0
    mean_tau = taus.mean(axis=0)
    sd_tau   = taus.std(axis=0, ddof=1) if done > 1 else np.zeros_like(mean_tau)
    bias_tau = mean_tau - tau0
    avg_se   = ses.mean(axis=0)
    cov_pct  = covs.mean(axis=0) * 100.0

    out_rows = []
    for j, m in enumerate(model_names):
        out_rows.append({
            "Design": design,
            "N": n,
            "Model": m,
            "Mean(tau_hat)": float(mean_tau[j]),
            "Bias(tau_hat)": float(bias_tau[j]),
            "SD(tau_hat)": float(sd_tau[j]),
            "Avg(SE)": float(avg_se[j]),
            "Coverage(%)": float(cov_pct[j]),
            "Bias(q)": np.nan, "Var(q)": np.nan, "MSE(q)": np.nan,
        })

    if do_q_diag:
        mean_Q = Q_sum / done
        var_Q_pointwise = (Q_sum2 / done) - mean_Q**2
        bias2 = np.mean((mean_Q - true_q_eval[None, :])**2, axis=1)
        var_  = np.mean(var_Q_pointwise, axis=1)
        mse_  = bias2 + var_
        for j in range(len(model_names)):
            out_rows[j]["Bias(q)"] = float(np.sqrt(bias2[j]))
            out_rows[j]["Var(q)"]  = float(var_[j])
            out_rows[j]["MSE(q)"]  = float(mse_[j])

    return out_rows, done

# -------------------------------------------------------------------
# MAIN
# -------------------------------------------------------------------
if __name__ == "__main__":
    print("=======================================================")
    print("TEJ-safe MC: Adaptive Nested-CV DML-IV (optimized)")
    print("=======================================================")
    print(f"HAS_XGB={HAS_XGB}")
    print(f"N_SIMS={N_SIMS}, K_OUTER={K_OUTER}, K_INNER={K_INNER}")
    print(f"Parallel: MAX_WORKERS={MAX_WORKERS}, N_INNER_JOBS={N_INNER_JOBS}, TREE_N_JOBS={TREE_N_JOBS}")
    print(f"Checkpoint root: {OUT_ROOT.resolve()}")
    print(f"Q diagnostics: {DO_Q_DECOMP_DIAGNOSTICS} (N_EVAL={N_EVAL})")

    lib0 = get_mc_library()
    model_names = [m[0] for m in lib0] + ["Adaptive Nested-CV"]

    all_results = []

    for design in DESIGNS:
        for n in N_SAMPLES:
            print(f"\n=== Running cell: design={design} | N={n} ===")

            missing = [sim for sim in range(N_SIMS) if not sim_outfile(design, n, sim).exists()]
            if len(missing) == 0:
                print("  All sims already completed (skipping run).")
            else:
                print(f"  Missing sims: {len(missing)}/{N_SIMS} -> running with {MAX_WORKERS} workers...")
                Parallel(n_jobs=MAX_WORKERS)(
                    delayed(_run_one_sim_and_save)(design, n, sim, K_OUTER, K_INNER, DO_Q_DECOMP_DIAGNOSTICS, N_EVAL)
                    for sim in missing
                )
                print("  Finished running missing sims.")

            rows, done = aggregate_cell(design, n, model_names, DO_Q_DECOMP_DIAGNOSTICS, N_EVAL)
            print(f"  Aggregated {done} completed sims.")
            all_results.extend(rows)

    results_df = pd.DataFrame(all_results)

    for design in DESIGNS:
        print(f"\n\n==================================================================")
        print(f"   MONTE CARLO RESULTS: {design.upper()} DGP")
        print(f"==================================================================")

        design_df = results_df[results_df["Design"] == design].drop(columns=["Design"])

        # <-- FIX 3: only export TABLE 1 if diagnostics are enabled
        if DO_Q_DECOMP_DIAGNOSTICS:
            print("\n--- TABLE 1: First-Stage Prediction Metrics (q = E[D|Z,W]) ---")
            first_stage = design_df.pivot(index="Model", columns="N", values=["Bias(q)", "Var(q)", "MSE(q)"])
            first_stage.columns = [f"{col[0]}_{col[1]}" for col in first_stage.columns]
            first_stage = first_stage.round(4)
            print(first_stage)
            export_latex_table_safe(
                first_stage,
                f"tab_mc_first_stage_{design}.tex",
                f"Reduced-form first-stage prediction metrics for $q_0(Z,W)=E[D\\mid Z,W]$ ({design.capitalize()} DGP)",
                f"tab:mc_first_stage_{design}"
            )
        else:
            print("\n(Skipping TABLE 1 because DO_Q_DECOMP_DIAGNOSTICS=False)")

        print("\n--- TABLE 2: Second-Stage Causal Metrics ---")
        second_stage = design_df.pivot(
            index="Model", columns="N",
            values=["Mean(tau_hat)", "Bias(tau_hat)", "SD(tau_hat)", "Avg(SE)", "Coverage(%)"]
        )
        second_stage.columns = [f"{col[0]}_{col[1]}" for col in second_stage.columns]
        second_stage = second_stage.round(4)
        print(second_stage)

        export_latex_table_safe(
            second_stage,
            f"tab_mc_second_stage_{design}.tex",
            f"Second-stage DML-IV performance metrics ({design.capitalize()} DGP)",
            f"tab:mc_second_stage_{design}"
        )

    print("\nDone. LaTeX tables exported.")

TEJ-safe MC: Adaptive Nested-CV DML-IV (optimized)
HAS_XGB=True
N_SIMS=100, K_OUTER=3, K_INNER=3
Parallel: MAX_WORKERS=4, N_INNER_JOBS=1, TREE_N_JOBS=1
Checkpoint root: /Users/TC/PLIV DML for ECONOMETRICS Journal/mc_out
Q diagnostics: False (N_EVAL=5000)

=== Running cell: design=sparse | N=1000 ===
  Missing sims: 72/100 -> running with 4 workers...
  Finished running missing sims.
  Aggregated 100 completed sims.

=== Running cell: design=sparse | N=3000 ===
  Missing sims: 100/100 -> running with 4 workers...


/Users/TC/anaconda3/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/TC/anaconda3/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/TC/anaconda3/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/TC/anaconda3/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/TC/anaconda3/lib/python3.11/site-packages

  Finished running missing sims.
  Aggregated 100 completed sims.

=== Running cell: design=sparse | N=10000 ===
  Missing sims: 100/100 -> running with 4 workers...
  Finished running missing sims.
  Aggregated 100 completed sims.

=== Running cell: design=nonlinear | N=1000 ===
  Missing sims: 100/100 -> running with 4 workers...
  Finished running missing sims.
  Aggregated 100 completed sims.

=== Running cell: design=nonlinear | N=3000 ===
  Missing sims: 100/100 -> running with 4 workers...


/Users/TC/anaconda3/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/TC/anaconda3/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/TC/anaconda3/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/TC/anaconda3/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/TC/anaconda3/lib/python3.11/site-packages

  Finished running missing sims.
  Aggregated 100 completed sims.

=== Running cell: design=nonlinear | N=10000 ===
  Missing sims: 100/100 -> running with 4 workers...
  Finished running missing sims.
  Aggregated 100 completed sims.


   MONTE CARLO RESULTS: SPARSE DGP

(Skipping TABLE 1 because DO_Q_DECOMP_DIAGNOSTICS=False)

--- TABLE 2: Second-Stage Causal Metrics ---
                    Mean(tau_hat)_1000  Mean(tau_hat)_3000  \
Model                                                        
Adaptive Nested-CV              1.0323              0.9965   
ElasticNet                      1.0316              0.9963   
GradientBoosting                1.0289              0.9954   
Lasso                           1.0316              0.9970   
MLP                             0.9944              0.9327   
RandomForest                    1.0222              1.0002   
Ridge                           1.0450              0.9972   
XGBoost                         1.0247              0.9998   

   

In [17]:
from pathlib import Path
import numpy as np
import pandas as pd

OUT_ROOT = Path("mc_out")
DESIGNS = ["sparse", "nonlinear"]
N_SAMPLES = [1000, 3000, 10000]
N_SIMS = 100
TAU0 = 1.0

def sim_outfile(design, n, sim):
    return OUT_ROOT / design / f"n{n}" / f"sim_{sim:04d}.npz"

all_rows = []

for design in DESIGNS:
    for n in N_SAMPLES:
        taus, ses, covs = [], [], []
        for sim in range(N_SIMS):
            f = sim_outfile(design, n, sim)
            if not f.exists():
                continue
            dat = np.load(f, allow_pickle=True)
            model_names = list(dat["model_names"])
            taus.append(dat["tau"])
            ses.append(dat["se"])
            covs.append(dat["cov"])

        taus = np.vstack(taus)
        ses = np.vstack(ses)
        covs = np.vstack(covs)

        for j, m in enumerate(model_names):
            tau_j = taus[:, j].astype(float)
            se_j = ses[:, j].astype(float)
            cov_j = covs[:, j].astype(float)

            rmse = np.sqrt(np.mean((tau_j - TAU0) ** 2))
            avg_ci_len = np.mean(2 * 1.96 * se_j)

            all_rows.append({
                "Design": design,
                "N": n,
                "Model": m,
                "Mean(tau_hat)": np.mean(tau_j),
                "Bias(tau_hat)": np.mean(tau_j) - TAU0,
                "SD(tau_hat)": np.std(tau_j, ddof=1),
                "RMSE(tau_hat)": rmse,
                "Avg(SE)": np.mean(se_j),
                "Avg(CI length)": avg_ci_len,
                "Coverage(%)": 100.0 * np.mean(cov_j),
            })

df = pd.DataFrame(all_rows)
print(df)

# optional pivot for a compact LaTeX table
tab = df.pivot(index="Model", columns=["Design", "N"],
               values=["RMSE(tau_hat)", "Coverage(%)", "Avg(SE)"])
tab.columns = [f"{a}_{b}_{c}" for a,b,c in tab.columns]
tab = tab.round(4)
print(tab)

       Design      N               Model  Mean(tau_hat)  Bias(tau_hat)  \
0      sparse   1000               Ridge       1.045001       0.045001   
1      sparse   1000               Lasso       1.031571       0.031571   
2      sparse   1000          ElasticNet       1.031597       0.031597   
3      sparse   1000        RandomForest       1.022163       0.022163   
4      sparse   1000    GradientBoosting       1.028939       0.028939   
5      sparse   1000             XGBoost       1.024741       0.024741   
6      sparse   1000                 MLP       0.994439      -0.005561   
7      sparse   1000  Adaptive Nested-CV       1.032339       0.032339   
8      sparse   3000               Ridge       0.997238      -0.002762   
9      sparse   3000               Lasso       0.996992      -0.003008   
10     sparse   3000          ElasticNet       0.996318      -0.003682   
11     sparse   3000        RandomForest       1.000215       0.000215   
12     sparse   3000    GradientBoosti